In [ ]:
# In[1]:

import pandas as pd

# Load files
metric_csv = "metric.csv"
trace_csv = "trace.csv"
log_csv = "log.csv"
error_logs_csv = "error_logs.csv"

mdf = pd.read_csv(metric_csv)
tdf = pd.read_csv(trace_csv)
ldf = pd.read_csv(log_csv)
edf = pd.read_csv(error_logs_csv)

# Parse timestamps to UTC datetime (Unix seconds)
mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)
tdf['timestamp'] = pd.to_datetime(tdf['timestamp'], unit='s', utc=True)
ldf['timestamp'] = pd.to_datetime(ldf['timestamp'], unit='s', utc=True)
edf['timestamp'] = pd.to_datetime(edf['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates: group by cmdb_id & kpi_name
metric_counts = mdf.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('count')
metric_min = mdf.groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().rename('min_timestamp')
metric_max = mdf.groupby(['cmdb_id', 'kpi_name'])['timestamp'].max().rename('max_timestamp')
metric_p95 = mdf.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_value')

metric_summary = (
    pd.concat([metric_counts, metric_min, metric_max, metric_p95], axis=1)
      .reset_index()
      .sort_values('count', ascending=False)
      .head(20)
)

# 2) trace.csv distinct trace_name with counts and min/max timestamps (top 20 by count)
trace_summary = (
    tdf.groupby('trace_name')['timestamp']
       .agg(count='count', min_timestamp='min', max_timestamp='max')
       .reset_index()
       .sort_values('count', ascending=False)
       .head(20)
)

# 3) log.csv distinct log_name with counts and min/max timestamps (top 20 by count)
log_summary = (
    ldf.groupby('log_name')['timestamp']
       .agg(count='count', min_timestamp='min', max_timestamp='max')
       .reset_index()
       .sort_values('count', ascending=False)
       .head(20)
)

# 4) error_logs.csv: total count, min/max timestamp, top 10 cmdb_id by count
error_total_count = len(edf)
error_min_ts = edf['timestamp'].min() if error_total_count > 0 else pd.NaT
error_max_ts = edf['timestamp'].max() if error_total_count > 0 else pd.NaT
error_top_cmdb = (
    edf.groupby('cmdb_id').size().rename('count')
       .reset_index()
       .sort_values('count', ascending=False)
       .head(10)
)

# Compact summary table for error logs (1-row)
error_summary = pd.DataFrame([{
    'total_count': error_total_count,
    'min_timestamp': error_min_ts,
    'max_timestamp': error_max_ts
}])

# Display results (compact)
metric_summary, trace_summary, log_summary, error_summary, error_top_cmdb

```
Out[1]:
```
```python
# Compose a plain-English summary string based on previously computed summaries:
metric_top = metric_summary.copy()
metric_time_min = metric_top['min_timestamp'].min()
metric_time_max = metric_top['max_timestamp'].max()

# Prepare a short list of top metric groups (up to 8) with key fields
rows = []
for _, r in metric_top.head(8).iterrows():
    rows.append(f"{r['cmdb_id']} - {r['kpi_name']} (count={int(r['count'])}, p95={r['p95_value']:.6g}, "
                f"{r['min_timestamp'].isoformat()} to {r['max_timestamp'].isoformat()})")

trace_empty = trace_summary.shape[0] == 0
log_rows = []
for _, r in log_summary.iterrows():
    log_rows.append(f"{r['log_name']} (count={int(r['count'])}, {r['min_timestamp'].isoformat()} to {r['max_timestamp'].isoformat()})")

error_count = int(error_summary.at[0, 'total_count'])
error_top = error_top_cmdb.copy()

summary = (
    "Summary of telemetry (plain English):\n\n"
    f"Time range for metric data shown: {metric_time_min.isoformat()} to {metric_time_max.isoformat()} (UTC).\n\n"
    "1) Metrics (top groups by sample count):\n"
    + ("\n".join(f" - {x}" for x in rows)) + "\n\n"
    "Observations:\n"
    " - Metrics include multiple KPIs for carts, carts-db, catalogue, and catalogue-db. Counts shown are per-group (most are 25 samples each).\n"
    " - Notable p95 values in the top groups: high memory p95 for catalogue-db and carts (e.g., catalogue-db mem p95 ~4.31e8, carts mem p95 ~2.07e8), and elevated diskio p95 for carts-db (~2.28e6).\n\n"
    "2) Traces:\n"
    + (" - No trace records available in trace.csv." if trace_empty else " - " + ", ".join(
        f"{r['trace_name']} (count={int(r['count'])})" for _, r in trace_summary.head(20).iterrows()
      )) + "\n\n"
    "3) Logs:\n"
    + (" - " + "\n - ".join(log_rows) + "\n\n" if log_rows else " - No log records available.\n\n")
    "4) Error logs:\n"
    f" - Total error_logs rows: {error_count}.\n"
    + (f" - Top cmdb_id by error count (top 10):\n" +
       "\n".join(f\"   - {row['cmdb_id']}: {int(row['count'])}\" for _, row in error_top.iterrows())
       if not error_top.empty else " - No error log entries by cmdb_id.\n")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count             min_timestamp             max_timestamp     p95_value
0          carts         cpu     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.499647e+00
1          carts  latency-50     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.840076e-02
2          carts  latency-90     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.142660e-02
3          carts         mem     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  2.074342e+08
4          carts      socket     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.645000e+01
5          carts    workload     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  8.517723e+00
6       carts-db         cpu     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.945813e+00
7       carts-db      diskio     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  2.276567e+06
8       carts-db         mem     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  8.743823e+07
9       carts-db      socket     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.000000e+01
10     catalogue         cpu     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.721597e-01
11     catalogue  latency-50     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  3.049924e-03
12     catalogue  latency-90     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.689864e-03
13     catalogue         mem     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  6.187332e+06
14     catalogue      socket     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  7.000000e+00
15     catalogue    workload     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.232523e+00
16  catalogue-db         cpu     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  1.972620e-01
17  catalogue-db      diskio     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  0.000000e+00
18  catalogue-db         mem     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.308722e+08
19  catalogue-db      socket     25 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00  4.000000e+00, Empty DataFrame
Columns: [trace_name, count, min_timestamp, max_timestamp]
Index: [],           log_name  count             min_timestamp             max_timestamp
0  log.error_count    232 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00
1    log.row_count    232 2024-01-21 08:28:00+00:00 2024-01-21 08:52:00+00:00,    total_count min_timestamp max_timestamp
0            0           NaT           NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric dataframe 'mdf' if present; otherwise load it.
try:
    mdf
except NameError:
    mdf = pd.read_csv("metric.csv")
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
if not pd.api.types.is_datetime64tz_dtype(mdf['timestamp']):
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) on the full series (global threshold)
global_p95 = (
    mdf.groupby(['cmdb_id', 'kpi_name'])['value']
       .quantile(0.95)
       .rename('global_p95')
       .reset_index()
)

# 2) Filter points to the incident window (UTC)
window_start = pd.to_datetime("2024-01-21 08:25:46", utc=True)
window_end   = pd.to_datetime("2024-01-21 08:55:46", utc=True)
mdf_window = mdf[(mdf['timestamp'] >= window_start) & (mdf['timestamp'] <= window_end)].copy()

# 3) Attach global_p95 to windowed points
mdf_win_with_p95 = mdf_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value > global_p95 (strictly greater as requested)
mdf_win_with_p95['is_anomaly'] = mdf_win_with_p95['value'] > mdf_win_with_p95['global_p95']

# Aggregate per group
points_in_window = (
    mdf_win_with_p95.groupby(['cmdb_id', 'kpi_name'])['value']
    .count()
    .rename('points_in_window')
)

anomaly_count = (
    mdf_win_with_p95.groupby(['cmdb_id', 'kpi_name'])['is_anomaly']
    .sum()
    .astype(int)
    .rename('anomaly_count')
)

max_value_in_window = (
    mdf_win_with_p95.groupby(['cmdb_id', 'kpi_name'])['value']
    .max()
    .rename('max_value_in_window')
)

earliest_anomaly = (
    mdf_win_with_p95[mdf_win_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_timestamp')
)

# Combine aggregates into one DataFrame
agg = (
    pd.concat([points_in_window, anomaly_count, max_value_in_window], axis=1)
      .reset_index()
      .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
      .merge(earliest_anomaly.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')
)

# Keep only groups with at least one anomaly
anomalies_summary = (
    agg[agg['anomaly_count'] > 0]
      .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count',
               'earliest_anomaly_timestamp', 'max_value_in_window', 'points_in_window']]
      .sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
      .head(20)
      .reset_index(drop=True)
)

# Display compact table of anomaly groups (up to 20 rows)
anomalies_summary

```
Out[2]:
```
```python
# Compose a concise plain-English summary string from the existing anomaly summary dataframe
try:
    df = anomalies_summary
except NameError:
    # If anomalies_summary is not present, load and recompute quickly (fallback - should not usually trigger)
    import pandas as pd
    mdf = pd.read_csv("metric.csv")
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)
    global_p95 = mdf.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_p95').reset_index()
    window_start = pd.to_datetime("2024-01-21 08:25:46", utc=True)
    window_end   = pd.to_datetime("2024-01-21 08:55:46", utc=True)
    mdf_window = mdf[(mdf['timestamp'] >= window_start) & (mdf['timestamp'] <= window_end)]
    mdf_win_with_p95 = mdf_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
    mdf_win_with_p95['is_anomaly'] = mdf_win_with_p95['value'] > mdf_win_with_p95['global_p95']
    points_in_window = mdf_win_with_p95.groupby(['cmdb_id','kpi_name'])['value'].count().rename('points_in_window')
    anomaly_count = mdf_win_with_p95.groupby(['cmdb_id','kpi_name'])['is_anomaly'].sum().astype(int).rename('anomaly_count')
    max_value_in_window = mdf_win_with_p95.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window')
    earliest_anomaly = mdf_win_with_p95[mdf_win_with_p95['is_anomaly']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('earliest_anomaly_timestamp')
    agg = pd.concat([points_in_window, anomaly_count, max_value_in_window], axis=1).reset_index().merge(global_p95, on=['cmdb_id','kpi_name'], how='left').merge(earliest_anomaly.reset_index(), on=['cmdb_id','kpi_name'], how='left')
    df = agg[agg['anomaly_count'] > 0].loc[:, ['cmdb_id','kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','points_in_window']].sort_values(['anomaly_count','max_value_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

total_groups = df.shape[0]
unique_anom_counts = sorted(df['anomaly_count'].unique())
unique_points_in_window = sorted(df['points_in_window'].unique())

# Prepare top 5 details
top_rows = []
for _, r in df.head(5).iterrows():
    top_rows.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']:.6g} | "
        f"anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp'].isoformat()} | "
        f"max_in_window={r['max_value_in_window']:.6g} | points_in_window={int(r['points_in_window'])}"
    )

summary = (
    f"Anomaly summary (metric.csv) for window 2024-01-21 08:25:46 to 2024-01-21 08:55:46 UTC:\n\n"
    f"- Groups with at least one anomaly: {total_groups} (showing up to 20 groups).\n"
    f"- Observed anomaly counts in these groups: {unique_anom_counts} (most groups have 2 anomalies each).\n"
    f"- Points per group within the window: {unique_points_in_window} (most groups show 25 points in-window).\n\n"
    "Top 5 anomaly groups (cmdb_id | kpi_name | global_p95 | anomalies | earliest_anomaly | max_in_window | points_in_window):\n"
    + "\n".join(f" - {r}" for r in top_rows) + "\n\n"
    "Overall observations:\n"
    "- Memory (mem) anomalies are widespread across many services (user, catalogue-db, orders, shipping, carts, front-end, rabbitmq, etc.).\n"
    "- Some disk I/O anomalies appear for DB and queue components (e.g., carts-db, user-db, orders-db, queue-master, session-db).\n"
    "- A few non-memory anomalies are present (e.g., user cpu, orders socket).\n"
    "- Each listed group exceeded its global P95 at least once within the incident window; the max values shown are the peak observed in-window.\n\n"
    "Recommendation: prioritize investigation on services showing high memory peaks (especially 'user' and 'catalogue-db') and the DB/queue disk I/O spikes, as these are the most prominent anomalies in the window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  points_in_window
0           user      mem  6.299988e+08              2  2024-01-21 08:44:00+00:00         6.458502e+08                25
1   catalogue-db      mem  4.308722e+08              2  2024-01-21 08:50:00+00:00         4.308746e+08                25
2         orders      mem  3.331894e+08              2  2024-01-21 08:46:00+00:00         3.332364e+08                25
3       shipping      mem  3.071957e+08              2  2024-01-21 08:46:00+00:00         3.071983e+08                25
4          carts      mem  2.074342e+08              2  2024-01-21 08:33:00+00:00         2.075106e+08                25
5      front-end      mem  1.049263e+08              2  2024-01-21 08:47:00+00:00         1.059185e+08                25
6       rabbitmq      mem  9.535357e+07              2  2024-01-21 08:51:00+00:00         9.595011e+07                25
7       carts-db      mem  8.743823e+07              2  2024-01-21 08:51:00+00:00         8.760900e+07                25
8        user-db      mem  5.296293e+07              2  2024-01-21 08:51:00+00:00         5.321079e+07                25
9      orders-db      mem  4.328384e+07              2  2024-01-21 08:51:00+00:00         4.341777e+07                25
10     catalogue      mem  6.187332e+06              2  2024-01-21 08:51:00+00:00         6.191377e+06                25
11       payment      mem  4.611867e+06              2  2024-01-21 08:49:00+00:00         4.612847e+06                25
12    session-db      mem  2.935603e+06              2  2024-01-21 08:33:00+00:00         2.976222e+06                25
13      carts-db   diskio  2.276567e+06              2  2024-01-21 08:34:00+00:00         2.373795e+06                25
14       user-db   diskio  1.688611e+06              2  2024-01-21 08:49:00+00:00         1.692844e+06                25
15     orders-db   diskio  9.193046e+05              2  2024-01-21 08:28:00+00:00         9.995896e+05                25
16  queue-master   diskio  5.211473e+04              2  2024-01-21 08:35:00+00:00         5.374840e+04                25
17    session-db   diskio  1.129895e+04              2  2024-01-21 08:40:00+00:00         1.423389e+04                25
18          user      cpu  3.942775e+01              2  2024-01-21 08:44:00+00:00         3.950382e+01                25
19        orders   socket  2.564546e+01              2  2024-01-21 08:46:00+00:00         2.628333e+01                25```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing log dataframe 'ldf' if available; otherwise load it
try:
    ldf
except NameError:
    ldf = pd.read_csv("log.csv")

# Ensure timestamp parsed as UTC datetime
if not pd.api.types.is_datetime64tz_dtype(ldf['timestamp']):
    ldf['timestamp'] = pd.to_datetime(ldf['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-21 08:25:46", utc=True)
window_end   = pd.to_datetime("2024-01-21 08:55:46", utc=True)

# Filter to window (inclusive)
ldf_win = ldf[(ldf['timestamp'] >= window_start) & (ldf['timestamp'] <= window_end)].copy()

# Group by cmdb_id and log_name and compute aggregations
grp = (
    ldf_win.groupby(['cmdb_id', 'log_name'])['value']
    .agg(count='count', mean_value='mean', max_value='max', sum_value='sum')
    .reset_index()
)

# Sort by mean_value descending and keep top 20 rows
grp_top20 = grp.sort_values('mean_value', ascending=False).head(20).reset_index(drop=True)

# Compute total sum of value for log.error_count across all cmdb_id within window
error_count_sum = ldf_win.loc[ldf_win['log_name'] == 'log.error_count', 'value'].sum()
error_count_total = pd.DataFrame([{'log_name': 'log.error_count', 'total_sum_value_in_window': error_count_sum}])

# Return compact results
grp_top20, error_count_total

```
Out[3]:
```
```python
# Compose a plain-English summary string from the previously computed results (reuse variables if available)
try:
    grp_top20
    error_count_total
except NameError:
    import pandas as pd
    ldf = pd.read_csv("log.csv")
    ldf['timestamp'] = pd.to_datetime(ldf['timestamp'], unit='s', utc=True)
    window_start = pd.to_datetime("2024-01-21 08:25:46", utc=True)
    window_end   = pd.to_datetime("2024-01-21 08:55:46", utc=True)
    ldf_win = ldf[(ldf['timestamp'] >= window_start) & (ldf['timestamp'] <= window_end)].copy()
    grp = (ldf_win.groupby(['cmdb_id', 'log_name'])['value']
           .agg(count='count', mean_value='mean', max_value='max', sum_value='sum')
           .reset_index())
    grp_top20 = grp.sort_values('mean_value', ascending=False).head(20).reset_index(drop=True)
    error_count_total = pd.DataFrame([{
        'log_name': 'log.error_count',
        'total_sum_value_in_window': ldf_win.loc[ldf_win['log_name'] == 'log.error_count', 'value'].sum()
    }])

# Build summary lines
window_str = "2024-01-21 08:25:46 to 2024-01-21 08:55:46 UTC"
top_lines = []
for _, r in grp_top20.head(10).iterrows():
    top_lines.append(
        f"{r['cmdb_id']} | {r['log_name']} | count={int(r['count'])} | mean={r['mean_value']:.2f} | "
        f"max={int(r['max_value']) if pd.notna(r['max_value']) else 'NA'} | sum={int(r['sum_value'])}"
    )

total_error_sum = int(error_count_total.at[0, 'total_sum_value_in_window'])

summary = (
    f"Log telemetry summary for window {window_str}:\n\n"
    "- Top log groups by mean(value) (up to 10 shown):\n"
    + "\n".join(" - " + line for line in top_lines) + "\n\n"
    f"- Total sum of 'log.error_count' across all services in the window: {total_error_sum}\n\n"
    "Key observations:\n"
    "- 'log.row_count' dominates the top entries (front-end highest mean ~1890.64, then user ~569.72, queue-master ~531.00, etc.).\n"
    "- All 'log.error_count' groups recorded zero sum/value in the window (total sum = 0), indicating no logged errors in this period.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  count   mean_value  max_value  sum_value
0      front-end    log.row_count     25  1890.640000       2145      47266
1           user    log.row_count     25   569.720000        654      14243
2   queue-master    log.row_count     25   531.000000        621      13275
3      catalogue    log.row_count     25   156.840000        170       3921
4         orders    log.row_count     25   118.000000        138       2950
5        payment    log.row_count     25    97.400000        109       2435
6          carts    log.row_count     24    62.666667        175       1504
7       shipping    log.row_count     25    59.000000         69       1475
8     session-db    log.row_count      5     5.000000          5         25
9        user-db    log.row_count      4     2.000000          5          8
10      carts-db    log.row_count     24     1.208333          3         29
11     catalogue  log.error_count     25     0.000000          0          0
12         carts  log.error_count     24     0.000000          0          0
13      carts-db  log.error_count     24     0.000000          0          0
14  queue-master  log.error_count     25     0.000000          0          0
15       payment  log.error_count     25     0.000000          0          0
16     front-end  log.error_count     25     0.000000          0          0
17        orders  log.error_count     25     0.000000          0          0
18      shipping  log.error_count     25     0.000000          0          0
19    session-db  log.error_count      5     0.000000          0          0,           log_name  total_sum_value_in_window
0  log.error_count                          0)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing metric dataframe 'mdf' if available; otherwise load it
try:
    mdf
except NameError:
    mdf = pd.read_csv("metric.csv")
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64tz_dtype(mdf['timestamp']):
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'], unit='s', utc=True)

# Target components
targets = ["carts", "catalogue", "orders", "payment", "user"]

# Compute global P95 per (cmdb_id, kpi_name) using full series but restricted to target cmdb_ids
full_groups = mdf[mdf['cmdb_id'].isin(targets)].copy()
global_p95 = (
    full_groups.groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .rename('global_p95')
    .reset_index()
)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-21 08:25:46", utc=True)
window_end   = pd.to_datetime("2024-01-21 08:55:46", utc=True)

# Filter points to window and targets
mdf_window = mdf[
    (mdf['cmdb_id'].isin(targets)) &
    (mdf['timestamp'] >= window_start) &
    (mdf['timestamp'] <= window_end)
].copy()

# Attach global_p95 to windowed points
mdf_win = mdf_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies (strictly greater than global P95)
mdf_win['is_anomaly'] = mdf_win['value'] > mdf_win['global_p95']

# Prepare grouped aggregations
grouped = mdf_win.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).groupby(['cmdb_id', 'kpi_name'])

# Basic aggregates
agg_basic = grouped['value'].agg(
    points_in_window='count',
    anomaly_count=lambda s: (s > mdf_win.loc[s.index, 'global_p95']).sum()  # fallback, will be corrected below
).reset_index()

# More robust aggregates using the merged frame columns
points_in_window = grouped['value'].count().rename('points_in_window').reset_index()
anomaly_count = grouped['is_anomaly'].sum().astype(int).rename('anomaly_count').reset_index()
max_value_in_window = grouped['value'].max().rename('max_value_in_window').reset_index()
earliest_anomaly = (
    mdf_win[mdf_win['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_timestamp')
    .reset_index()
)

# Longest consecutive anomaly run per group
def longest_consecutive_run(df):
    s = df.sort_values('timestamp')['is_anomaly'].astype(int).values
    if s.size == 0:
        return 0
    max_run = 0
    cur = 0
    for v in s:
        if v == 1:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

longest_run = grouped.apply(longest_consecutive_run).rename('longest_consecutive_anomaly_run').reset_index()

# Combine aggregates
summary = (
    points_in_window
    .merge(anomaly_count, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(max_value_in_window, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anomaly, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(longest_run, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute percent_exceedance where anomaly_count > 0
summary['percent_exceedance'] = (
    (summary['max_value_in_window'] - summary['global_p95']) / summary['global_p95']
)
summary.loc[summary['anomaly_count'] == 0, 'percent_exceedance'] = pd.NA

# Keep only groups with anomaly_count > 0
anomalies = summary[summary['anomaly_count'] > 0].copy()

# Select and order columns as requested
anomalies = anomalies[
    ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_in_window', 'percent_exceedance', 'longest_consecutive_anomaly_run', 'points_in_window']
]

# Sort by percent_exceedance desc then anomaly_count desc and limit to top 20
anomalies_sorted = anomalies.sort_values(
    by=['percent_exceedance', 'anomaly_count'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Display compact result
anomalies_sorted

```
Out[4]:
```
```python
# Compose a concise plain-English summary from the existing anomalies_sorted DataFrame
try:
    df = anomalies_sorted
except NameError:
    raise RuntimeError("Required dataframe 'anomalies_sorted' not found in the kernel state.")

total_groups = int(df.shape[0])
window = "2024-01-21 08:25:46 to 2024-01-21 08:55:46 UTC"

# Prepare top 5 highlights
top5_lines = []
for _, r in df.head(5).iterrows():
    top5_lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']:.6g} | "
        f"anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp'].isoformat()} | "
        f"max_in_window={r['max_value_in_window']:.6g} | percent_exceedance={r['percent_exceedance']:.3f} | "
        f"longest_run={int(r['longest_consecutive_anomaly_run'])}"
    )

summary = (
    f"Anomaly analysis (restricted to components: carts, catalogue, orders, payment, user)\n"
    f"Incident window: {window}\n\n"
    f"- Groups with anomalies (value > group global P95): {total_groups} groups (limited to top 20 shown).\n"
    f"- Most groups have 2 anomaly points within the window and 25 points total in-window.\n\n"
    "Top 5 anomaly groups by percent exceedance:\n"
    + "\n".join(" - " + line for line in top5_lines) + "\n\n"
    "Key observations:\n"
    "- The largest relative spike is payment | cpu (percent_exceedance ~17.65), with 2 anomaly samples and the longest run = 2 — this is the most prominent outlier.\n"
    "- Many anomalies are latency-related (payment/user/orders latency-90 and latency-50) and moderate CPU/workload/socket elevations across services.\n"
    "- Longest consecutive anomaly runs are short (mostly 1, occasional 2), indicating brief spikes rather than prolonged continuous breaches.\n\n"
    "Recommendation: prioritize investigation of the payment CPU spike and the latency spikes in user/orders, then review CPU/workload/socket patterns for the other affected services."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  percent_exceedance  longest_consecutive_anomaly_run  points_in_window
0     payment         cpu  9.516994e-01              2  2024-01-21 08:28:00+00:00         1.774964e+01           17.650466                                2                25
1     payment  latency-90  4.496723e-03              2  2024-01-21 08:28:00+00:00         8.255442e-03            0.835879                                1                25
2        user  latency-90  1.304026e-01              2  2024-01-21 08:47:00+00:00         1.810894e-01            0.388695                                1                25
3        user  latency-50  1.185156e-02              2  2024-01-21 08:47:00+00:00         1.633598e-02            0.378382                                1                25
4      orders  latency-90  1.663696e-01              2  2024-01-21 08:46:00+00:00         1.949923e-01            0.172043                                1                25
5      orders         cpu  2.331794e+00              2  2024-01-21 08:43:00+00:00         2.690066e+00            0.153647                                1                25
6       carts         cpu  1.499647e+00              2  2024-01-21 08:31:00+00:00         1.665703e+00            0.110730                                1                25
7     payment    workload  2.250153e+00              2  2024-01-21 08:28:00+00:00         2.361714e+00            0.049579                                1                25
8       carts      socket  1.645000e+01              2  2024-01-21 08:47:00+00:00         1.720000e+01            0.045593                                1                25
9      orders    workload  2.270783e+00              2  2024-01-21 08:28:00+00:00         2.357143e+00            0.038031                                1                25
10    payment  latency-50  2.036602e-03              2  2024-01-21 08:47:00+00:00         2.098883e-03            0.030581                                2                25
11     orders  latency-50  4.627816e-02              2  2024-01-21 08:45:00+00:00         4.764136e-02            0.029457                                1                25
12      carts    workload  8.517723e+00              2  2024-01-21 08:42:00+00:00         8.746667e+00            0.026878                                1                25
13       user         mem  6.299988e+08              2  2024-01-21 08:44:00+00:00         6.458502e+08            0.025161                                1                25
14     orders      socket  2.564546e+01              2  2024-01-21 08:46:00+00:00         2.628333e+01            0.024873                                1                25
15  catalogue    workload  4.232523e+00              2  2024-01-21 08:42:00+00:00         4.336900e+00            0.024661                                1                25
16       user    workload  1.965108e+01              2  2024-01-21 08:28:00+00:00         2.009017e+01            0.022344                                1                25
17      carts  latency-90  4.142660e-02              2  2024-01-21 08:50:00+00:00         4.192554e-02            0.012044                                1                25
18  catalogue         cpu  1.721597e-01              2  2024-01-21 08:36:00+00:00         1.737811e-01            0.009418                                1                25
19       user      socket  3.800000e+01              1  2024-01-21 08:52:00+00:00         3.817021e+01            0.004479                                1                25```
```